In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
def popularidad_categoria(streams):
    if streams is None:
        return "Desconocida"
    elif streams < 50000:
        return "Baja"
    elif 50000 <= streams < 500000:
        return "Media"
    else:
        return "Alta"

In [0]:
pop_udf = F.udf(popularidad_categoria, StringType())

In [0]:
df_tracks = spark.table(f"{catalogo}.{esquema_source}.spotify_tracks")
df_charts = spark.table(f"{catalogo}.{esquema_source}.spotify_daily_charts") \
                 .withColumnRenamed("track_id", "chart_track_id")

In [0]:
df_tracks = df_tracks.dropna(how="all") \
                     .filter(col("track_id").isNotNull())

df_charts = df_charts.dropna(how="all") \
                     .filter(col("chart_track_id").isNotNull())

In [0]:
df_charts = df_charts.withColumn("popularity_category", pop_udf("streams"))

In [0]:
df_joined = df_charts.alias("c").join(
    df_tracks.alias("t"), 
    col("c.chart_track_id") == col("t.track_id"), 
    "inner"
)

In [0]:
df_filtered_sorted = df_joined.filter(col("snapshot_date") > "2023-01-01").orderBy("rank")

In [0]:
df_filtered_sorted = df_filtered_sorted.withColumn(
    "days_since_chart", 
    F.datediff(F.current_date(), F.col("snapshot_date"))
)

In [0]:
df_aggregated = df_filtered_sorted.groupBy("country", "artist_name").agg(
    F.count("chart_track_id").alias("appearances_on_charts")
)

In [0]:
df_with_bpm_diff = df_filtered_sorted

In [0]:
df_updated = df_with_bpm_diff.select("*",
                                    when(col("country").isin("US", "GB", "CA"), lit("Major Market")).otherwise("Global Market").alias("market_type"),
                                    when(col("duration_ms") > 240000, lit("Long Track")).otherwise(lit("Standard Track")).alias("track_duration_type")
                                ).drop("chart_track_id", "ingestion_date")

In [0]:
df_updated.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema_sink}.spotify_transformed")